In [1]:

# ===============================================================
# ALTA10 — TURBINE WAKE-DIRECTION DATASETS
#
# Creates:
# 1. Direction lookup: every turbine x every 5 degrees
# 2. Direction lists: LOW / MODERATE / HIGH for each turbine
# 3. Historical observations: every mast timestamp x turbine
#
# Includes Jensen and Gaussian-style predictions.
# Uses existing model calculations, without modifying them.
#
# LOW:      deficit < 2%
# MODERATE: 2% <= deficit <= 15%
# HIGH:     deficit > 15%
# ===============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# ---------------------------------------------------------------
# 1. Check existing wake model results
# ---------------------------------------------------------------

required = [
    "lookups",
    "names",
    "DIRECTIONS",
    "max_direction_offset_wake",
    "mast",
    "OUTPUT"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the integrated farm-wide wake analysis first. "
        f"Missing: {missing}"
    )

SAVE_DIR = Path(OUTPUT)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LOW_LIMIT = 2.0
HIGH_LIMIT = 15.0

turbines = np.asarray(names)
directions = np.asarray(DIRECTIONS, dtype=int)

N_TURBINES = len(turbines)
N_DIRECTIONS = len(directions)

def classify_wake(values):
    values = np.asarray(values, dtype=float)
    return np.where(
        values < LOW_LIMIT,
        "LOW",
        np.where(
            values > HIGH_LIMIT,
            "HIGH",
            "MODERATE"
        )
    )

# ---------------------------------------------------------------
# 2. Build complete wind-direction lookup
# ---------------------------------------------------------------

lookup_tables = []

for model in ["jensen", "gaussian_style"]:

    if model not in lookups:
        raise KeyError(f"Model not found: {model}")

    # Shape: 72 directions x 47 turbines
    nominal = np.asarray(lookups[model])

    if nominal.shape != (N_DIRECTIONS, N_TURBINES):
        raise ValueError(
            f"Unexpected lookup shape for {model}: "
            f"{nominal.shape}"
        )

    # Maximum modeled deficit over offsets
    # -10, -5, 0, +5, +10 degrees.
    conservative = max_direction_offset_wake(
        nominal,
        directions
    )

    table = pd.DataFrame({
        "model": model,

        "turbine": np.tile(
            turbines, N_DIRECTIONS
        ),

        "wind_from_deg": np.repeat(
            directions, N_TURBINES
        ),

        "nominal_deficit_pct":
            nominal.reshape(-1),

        "max_offset_deficit_pct":
            conservative.reshape(-1),

        "wake_category":
            classify_wake(
                conservative.reshape(-1)
            )
    })

    lookup_tables.append(table)

wake_lookup = pd.concat(
    lookup_tables,
    ignore_index=True
)

wake_lookup = wake_lookup.sort_values(
    ["model", "turbine", "wind_from_deg"]
).reset_index(drop=True)

lookup_path = (
    SAVE_DIR / "ALTA10_wake_direction_lookup_5deg.csv"
)

wake_lookup.to_csv(
    lookup_path,
    index=False
)

print("\nCOMPLETE DIRECTION LOOKUP")
print(f"Rows: {len(wake_lookup):,}")
print("Saved:", lookup_path)

display(wake_lookup.head(15).round(2))

# ---------------------------------------------------------------
# 3. Create readable direction lists for each turbine
# ---------------------------------------------------------------

# Example:
# WTG159 | HIGH | 260, 265, 270, 275 ...
#
# Each value is the CENTER of a 5-degree direction bin.

direction_lists = (
    wake_lookup
    .groupby(
        ["model", "turbine", "wake_category"]
    )["wind_from_deg"]
    .apply(
        lambda s: ", ".join(
            f"{int(v)}°" for v in sorted(s)
        )
    )
    .unstack(fill_value="")
    .reset_index()
)

for category in ["LOW", "MODERATE", "HIGH"]:
    if category not in direction_lists:
        direction_lists[category] = ""

direction_lists = direction_lists[
    ["model", "turbine", "LOW", "MODERATE", "HIGH"]
]

lists_path = (
    SAVE_DIR / "ALTA10_wake_directions_by_turbine.csv"
)

direction_lists.to_csv(
    lists_path,
    index=False
)

print("\nDIRECTION LISTS BY TURBINE")

display(
    direction_lists[
        direction_lists.turbine.isin(
            ["WTG159", "WTG172"]
        )
    ]
)

# ---------------------------------------------------------------
# 4. Create a direction-category matrix (Jensen)
# ---------------------------------------------------------------

jensen_matrix = (
    wake_lookup[
        wake_lookup.model.eq("jensen")
    ]
    .pivot(
        index="turbine",
        columns="wind_from_deg",
        values="wake_category"
    )
)

jensen_matrix.columns = [
    f"{int(c):03d}deg"
    for c in jensen_matrix.columns
]

matrix_path = (
    SAVE_DIR / "ALTA10_Jensen_wake_category_matrix.csv"
)

jensen_matrix.to_csv(matrix_path)

# ---------------------------------------------------------------
# 5. Assign wake exposure to historical observations
# ---------------------------------------------------------------

# One row for each recorded mast timestamp and turbine.
# The mast direction is treated as the shared wind direction
# across the farm, just as in our prior analysis.

required_mast = [
    "timestamp",
    "mm_wind_direction_3",
    "mm_wind_speed_3"
]

missing_mast = [
    c for c in required_mast
    if c not in mast.columns
]

if missing_mast:
    raise KeyError(
        f"Mast dataset missing columns: {missing_mast}"
    )

observed = mast[
    required_mast
].copy()

observed["timestamp"] = pd.to_datetime(
    observed["timestamp"],
    utc=True,
    errors="coerce"
)

for c in [
    "mm_wind_direction_3",
    "mm_wind_speed_3"
]:
    observed[c] = pd.to_numeric(
        observed[c], errors="coerce"
    )

observed = observed.loc[
    observed.timestamp.notna()
    & observed.mm_wind_direction_3.between(0, 360)
    & observed.mm_wind_speed_3.between(3, 25)
].sort_values("timestamp")

if observed.timestamp.duplicated().any():
    raise ValueError(
        "Duplicate mast timestamps found."
    )

wind_angles = (
    observed.mm_wind_direction_3.to_numpy()
)

n_times = len(observed)

# Each timestamp gets one observation per turbine.
history_data = pd.DataFrame({

    "timestamp_utc": pd.to_datetime(
        np.repeat(
            observed.timestamp.to_numpy(),
            N_TURBINES
        ),
        utc=True
    ),

    "turbine": np.tile(
        turbines, n_times
    ),

    "mast_wind_from_deg": np.repeat(
        wind_angles,
        N_TURBINES
    ),

    "mast_wind_speed_ms": np.repeat(
        observed.mm_wind_speed_3.to_numpy(),
        N_TURBINES
    )
})

# Add predicted wake exposure from both models.
for model in ["jensen", "gaussian_style"]:

    exposure = max_direction_offset_wake(
        lookups[model],
        wind_angles
    )

    if exposure.shape != (
        n_times, N_TURBINES
    ):
        raise ValueError(
            "Historical wake array shape mismatch."
        )

    values = exposure.reshape(-1)

    history_data[
        f"{model}_wake_deficit_pct"
    ] = values

    history_data[
        f"{model}_wake_category"
    ] = classify_wake(values)

# Parquet is recommended for the larger timestamp-level table.
history_path = (
    SAVE_DIR
    / "ALTA10_historical_turbine_wake_observations.parquet"
)

history_data.to_parquet(
    history_path,
    index=False
)

print("\nHISTORICAL TURBINE WAKE OBSERVATIONS")
print(f"Rows: {len(history_data):,}")
print(f"Mast timestamps: {n_times:,}")
print(f"Turbines: {N_TURBINES}")
print("Saved:", history_path)

display(history_data.head(12).round(2))

# ---------------------------------------------------------------
# 6. Summary: share of directions in each category
# ---------------------------------------------------------------

direction_summary = (
    wake_lookup
    .groupby(
        ["model", "turbine", "wake_category"]
    )
    .size()
    .unstack(fill_value=0)
)

for category in ["LOW", "MODERATE", "HIGH"]:
    if category not in direction_summary:
        direction_summary[category] = 0

direction_summary = (
    direction_summary[
        ["LOW", "MODERATE", "HIGH"]
    ]
    .reset_index()
)

direction_summary = direction_summary.rename(
    columns={
        "LOW": "low_direction_bins",
        "MODERATE": "moderate_direction_bins",
        "HIGH": "high_direction_bins"
    }
)

for category in [
    "low", "moderate", "high"
]:
    direction_summary[
        f"{category}_pct_of_360deg"
    ] = (
        100
        * direction_summary[
            f"{category}_direction_bins"
        ]
        / N_DIRECTIONS
    )

summary_path = (
    SAVE_DIR
    / "ALTA10_wake_direction_coverage_summary.csv"
)

direction_summary.to_csv(
    summary_path,
    index=False
)

print("\nDIRECTION COVERAGE SUMMARY")

display(
    direction_summary.sort_values(
        ["model", "high_pct_of_360deg"],
        ascending=[True, False]
    ).head(15).round(2)
)

print("\n" + "=" * 65)
print("WAKE-DIRECTION DATASETS COMPLETE")
print("=" * 65)
print("\nSaved to:", SAVE_DIR)
print("\nFiles created:")
print("1.", lookup_path.name)
print("2.", lists_path.name)
print("3.", matrix_path.name)
print("4.", history_path.name)
print("5.", summary_path.name)


RuntimeError: Run the integrated farm-wide wake analysis first. Missing: ['lookups', 'names', 'DIRECTIONS', 'max_direction_offset_wake', 'mast', 'OUTPUT']

In [2]:

# ============================================================
# ALTA10 — OPEN AND EXPLORE ALL 5 WAKE DATASETS
# ============================================================

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

FOLDER = Path(
    r"C:\Users\alex-\OneDrive\Documents\GitHub"
    r"\CU-Capstone-Project\data\processed"
    r"\12_farmwide_wake_exposure\integrated_smooth_maps"
)

FILES = [
    "ALTA10_wake_direction_lookup_5deg.csv",
    "ALTA10_wake_directions_by_turbine.csv",
    "ALTA10_Jensen_wake_category_matrix.csv",
    "ALTA10_historical_turbine_wake_observations.parquet",
    "ALTA10_wake_direction_coverage_summary.csv"
]

for number, filename in enumerate(FILES, start=1):

    path = FOLDER / filename

    print("\n" + "=" * 75)
    print(f"DATASET {number}: {filename}")
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND — run the dataset creation cell first.")
        continue

    if path.suffix == ".csv":

        df_view = pd.read_csv(path)

        print(f"Rows: {len(df_view):,}")
        print(f"Columns: {len(df_view.columns)}")
        print("\nColumn names:")
        print(df_view.columns.tolist())

        # The matrix is wide, so show a manageable portion.
        if "category_matrix" in filename:
            display(df_view.iloc[:10, :13])

        # Show key turbines for long direction lists.
        elif "directions_by_turbine" in filename:
            selected = df_view[
                df_view["turbine"].isin(["WTG159", "WTG172"])
            ]
            display(selected)

        else:
            display(df_view.head(15))

    elif path.suffix == ".parquet":

        parquet = pq.ParquetFile(path)

        print(f"Rows: {parquet.metadata.num_rows:,}")
        print(f"Columns: {len(parquet.schema.names)}")

        print("\nColumn names:")
        print(parquet.schema.names)

        first_batch = next(
            parquet.iter_batches(batch_size=15)
        )

        display(first_batch.to_pandas())

print("\n" + "=" * 75)
print("ALL DATASETS REVIEWED")
print("=" * 75)

print("\nFolder:", FOLDER)



DATASET 1: ALTA10_wake_direction_lookup_5deg.csv
Rows: 6,768
Columns: 6

Column names:
['model', 'turbine', 'wind_from_deg', 'nominal_deficit_pct', 'max_offset_deficit_pct', 'wake_category']


,model,turbine,wind_from_deg,nominal_deficit_pct,max_offset_deficit_pct,wake_category
0,gaussian_style,WTG150,0,18.797194,30.737077,HIGH
1,gaussian_style,WTG150,5,24.124991,40.651518,HIGH
2,gaussian_style,WTG150,10,30.737077,54.399560,HIGH
3,gaussian_style,WTG150,15,40.651518,61.988223,HIGH
4,gaussian_style,WTG150,20,54.399560,61.988223,HIGH
5,gaussian_style,WTG150,25,61.988223,61.988223,HIGH
6,gaussian_style,WTG150,30,56.234715,61.988223,HIGH
7,gaussian_style,WTG150,35,46.734225,61.988223,HIGH
8,gaussian_style,WTG150,40,39.578013,56.234715,HIGH
9,gaussian_style,WTG150,45,33.553578,46.734225,HIGH



DATASET 2: ALTA10_wake_directions_by_turbine.csv
Rows: 94
Columns: 5

Column names:
['model', 'turbine', 'LOW', 'MODERATE', 'HIGH']


,model,turbine,LOW,MODERATE,HIGH
9,gaussian_style,WTG159,"0°, 5°, 10°, 15°, 20°, 25°, 30°, 290°, 295°, 3...","35°, 40°, 45°, 260°, 265°, 270°, 275°, 280°, 285°","50°, 55°, 60°, 65°, 70°, 75°, 80°, 85°, 90°, 9..."
21,gaussian_style,WTG172,"115°, 120°, 125°, 130°, 285°, 290°, 295°, 300°...","85°, 90°, 95°, 100°, 105°, 110°, 135°, 140°, 1...","0°, 5°, 10°, 15°, 20°, 25°, 30°, 35°, 40°, 45°..."
56,jensen,WTG159,"0°, 5°, 10°, 15°, 20°, 25°, 30°, 250°, 255°, 2...","35°, 40°, 45°, 185°","50°, 55°, 60°, 65°, 70°, 75°, 80°, 85°, 90°, 9..."
68,jensen,WTG172,"0°, 110°, 115°, 120°, 125°, 130°, 135°, 140°, ...","85°, 90°, 95°, 100°, 105°","5°, 10°, 15°, 20°, 25°, 30°, 35°, 40°, 45°, 50..."



DATASET 3: ALTA10_Jensen_wake_category_matrix.csv
Rows: 47
Columns: 73

Column names:
['turbine', '000deg', '005deg', '010deg', '015deg', '020deg', '025deg', '030deg', '035deg', '040deg', '045deg', '050deg', '055deg', '060deg', '065deg', '070deg', '075deg', '080deg', '085deg', '090deg', '095deg', '100deg', '105deg', '110deg', '115deg', '120deg', '125deg', '130deg', '135deg', '140deg', '145deg', '150deg', '155deg', '160deg', '165deg', '170deg', '175deg', '180deg', '185deg', '190deg', '195deg', '200deg', '205deg', '210deg', '215deg', '220deg', '225deg', '230deg', '235deg', '240deg', '245deg', '250deg', '255deg', '260deg', '265deg', '270deg', '275deg', '280deg', '285deg', '290deg', '295deg', '300deg', '305deg', '310deg', '315deg', '320deg', '325deg', '330deg', '335deg', '340deg', '345deg', '350deg', '355deg']


,turbine,000deg,005deg,010deg,015deg,020deg,025deg,030deg,035deg,040deg,045deg,050deg,055deg
0,WTG150,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
1,WTG151,LOW,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
2,WTG152,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
3,WTG153,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
4,WTG154,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
5,WTG155,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
6,WTG156,LOW,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
7,WTG157,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
8,WTG158,LOW,LOW,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH
9,WTG159,LOW,LOW,LOW,LOW,LOW,LOW,LOW,MODERATE,MODERATE,MODERATE,HIGH,HIGH



DATASET 4: ALTA10_historical_turbine_wake_observations.parquet
Rows: 979,104
Columns: 8

Column names:
['timestamp_utc', 'turbine', 'mast_wind_from_deg', 'mast_wind_speed_ms', 'jensen_wake_deficit_pct', 'jensen_wake_category', 'gaussian_style_wake_deficit_pct', 'gaussian_style_wake_category']


,timestamp_utc,turbine,mast_wind_from_deg,mast_wind_speed_ms,jensen_wake_deficit_pct,jensen_wake_category,gaussian_style_wake_deficit_pct,gaussian_style_wake_category
0,2026-05-01 00:00:00+00:00,WTG150,290.580294,9.220294,0.000000,LOW,0.437154,LOW
1,2026-05-01 00:00:00+00:00,WTG151,290.580294,9.220294,0.000000,LOW,0.767554,LOW
2,2026-05-01 00:00:00+00:00,WTG152,290.580294,9.220294,0.000000,LOW,1.298290,LOW
3,2026-05-01 00:00:00+00:00,WTG153,290.580294,9.220294,0.000000,LOW,0.933121,LOW
4,2026-05-01 00:00:00+00:00,WTG154,290.580294,9.220294,0.000000,LOW,1.107577,LOW
5,2026-05-01 00:00:00+00:00,WTG155,290.580294,9.220294,0.000000,LOW,1.121508,LOW
6,2026-05-01 00:00:00+00:00,WTG156,290.580294,9.220294,0.000000,LOW,0.754033,LOW
7,2026-05-01 00:00:00+00:00,WTG157,290.580294,9.220294,0.000000,LOW,1.590698,LOW
8,2026-05-01 00:00:00+00:00,WTG158,290.580294,9.220294,0.000000,LOW,0.946815,LOW
9,2026-05-01 00:00:00+00:00,WTG159,290.580294,9.220294,0.000000,LOW,1.570805,LOW



DATASET 5: ALTA10_wake_direction_coverage_summary.csv
Rows: 94
Columns: 8

Column names:
['model', 'turbine', 'low_direction_bins', 'moderate_direction_bins', 'high_direction_bins', 'low_pct_of_360deg', 'moderate_pct_of_360deg', 'high_pct_of_360deg']


,model,turbine,low_direction_bins,moderate_direction_bins,high_direction_bins,low_pct_of_360deg,moderate_pct_of_360deg,high_pct_of_360deg
0,gaussian_style,WTG150,41,8,23,56.944444,11.111111,31.944444
1,gaussian_style,WTG151,11,21,40,15.277778,29.166667,55.555556
2,gaussian_style,WTG152,4,21,47,5.555556,29.166667,65.277778
3,gaussian_style,WTG153,8,21,43,11.111111,29.166667,59.722222
4,gaussian_style,WTG154,6,14,52,8.333333,19.444444,72.222222
5,gaussian_style,WTG155,5,13,54,6.944444,18.055556,75.000000
6,gaussian_style,WTG156,10,13,49,13.888889,18.055556,68.055556
7,gaussian_style,WTG157,5,14,53,6.944444,19.444444,73.611111
8,gaussian_style,WTG158,10,12,50,13.888889,16.666667,69.444444
9,gaussian_style,WTG159,21,9,42,29.166667,12.500000,58.333333



ALL DATASETS REVIEWED

Folder: C:\Users\alex-\OneDrive\Documents\GitHub\CU-Capstone-Project\data\processed\12_farmwide_wake_exposure\integrated_smooth_maps


In [3]:

# =============================================================
# ALTA10 — MERGE WAKE-DIRECTION LOOKUP WITH SCADA
#
# Matches:
#   SCADA turbine + timestamp
#            ↓
#   Met mast wind direction at same timestamp
#            ↓
#   Nearest 5-degree wind-direction lookup
#            ↓
#   Jensen + Gaussian-style wake classification
#
# Outputs:
# 1. Merged SCADA + wake dataset (Parquet)
# 2. SCADA comparison by turbine and wake category (CSV)
# 3. Comparison restricted to mast speeds 7–10 m/s (CSV)
#
# No existing datasets are overwritten.
# =============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# -------------------------------------------------------------
# 1. File locations
# -------------------------------------------------------------

ROOT = Path(
    r"C:\Users\alex-\OneDrive\Documents\GitHub\CU-Capstone-Project"
)

SCADA_PATH = ROOT / "data/processed/scada_data.parquet"

MAST_PATH = ROOT / "data/processed/metmasts_data.parquet"

LOOKUP_PATH = (
    ROOT
    / "data/processed/12_farmwide_wake_exposure"
    / "integrated_smooth_maps"
    / "ALTA10_wake_direction_lookup_5deg.csv"
)

SAVE_DIR = (
    ROOT
    / "data/processed/12_farmwide_wake_exposure"
    / "scada_wake_comparison"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

for path in [SCADA_PATH, MAST_PATH, LOOKUP_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

# -------------------------------------------------------------
# 2. Helper functions
# -------------------------------------------------------------

def normalize_turbine(value):

    if pd.isna(value):
        return pd.NA

    match = re.fullmatch(
        r"(?:(?:WTG[_\s-]*T?|GE)[_\s-]*)?(\d+)(?:\.0)?",
        str(value).strip().upper()
    )

    if match is None:
        raise ValueError(
            f"Unrecognized turbine ID: {value!r}"
        )

    return f"WTG{int(match.group(1))}"


def prepare_time(frame, source_name):

    # Support timestamp or utc stored in the index.
    if not any(
        c in frame.columns for c in ["timestamp", "utc"]
    ):
        frame = frame.reset_index()

    time_col = next(
        (c for c in ["timestamp", "utc"] if c in frame.columns),
        None
    )

    if time_col is None:
        raise KeyError(
            f"{source_name}: timestamp or utc not found."
        )

    frame["timestamp"] = pd.to_datetime(
        frame[time_col],
        utc=True,
        errors="coerce"
    )

    if frame.timestamp.isna().any():
        raise ValueError(
            f"{source_name}: invalid timestamps detected."
        )

    return frame


def read_selected_parquet(path, preferred, required):

    # Read only necessary columns when possible to
    # avoid loading every field from the SCADA dataset.
    available = set(pq.read_schema(path).names)

    if not set(required).issubset(available) or not (
        {"timestamp", "utc"} & available
    ):
        return pd.read_parquet(path)

    selected = [
        c for c in preferred if c in available
    ]

    return pd.read_parquet(path, columns=selected)


# -------------------------------------------------------------
# 3. Load SCADA data
# -------------------------------------------------------------

scada = read_selected_parquet(
    SCADA_PATH,
    preferred=[
        "turbine",
        "timestamp",
        "utc",
        "wind_speed",
        "KPI_wind_speed",
        "real_power_ac",
        "KPI_real_power_ac",
        "air_density"
    ],
    required=["turbine", "wind_speed"]
)

if "turbine" not in scada.columns:
    scada = scada.reset_index()

scada = prepare_time(scada, "SCADA")

if "turbine" not in scada.columns:
    raise KeyError("SCADA turbine IDs not found.")

scada["turbine"] = scada.turbine.map(
    normalize_turbine
)

if scada.turbine.isna().any():
    raise ValueError("Missing SCADA turbine IDs.")

# Keep the important measurements, without replacing
# original power values using KPI zeros.
scada_columns = [
    "wind_speed",
    "KPI_wind_speed",
    "real_power_ac",
    "KPI_real_power_ac",
    "air_density"
]

for col in scada_columns:

    if col not in scada.columns:
        scada[col] = np.nan

    scada[col] = pd.to_numeric(
        scada[col],
        errors="coerce"
    )

scada = scada[
    ["timestamp", "turbine"] + scada_columns
].copy()

if scada.duplicated(
    ["turbine", "timestamp"]
).any():
    raise ValueError(
        "Duplicate turbine/timestamp records in SCADA."
    )

original_scada_rows = len(scada)

print("SCADA records:", f"{len(scada):,}")
print("Turbines:", scada.turbine.nunique())

# -------------------------------------------------------------
# 4. Load met-mast direction and wind speed
# -------------------------------------------------------------

mast = read_selected_parquet(
    MAST_PATH,
    preferred=[
        "timestamp",
        "utc",
        "mm_wind_direction_3",
        "mm_wind_speed_3"
    ],
    required=[
        "mm_wind_direction_3",
        "mm_wind_speed_3"
    ]
)

mast = prepare_time(mast, "Met mast")

for col in [
    "mm_wind_direction_3",
    "mm_wind_speed_3"
]:
    if col not in mast.columns:
        raise KeyError(f"Missing mast column: {col}")

    mast[col] = pd.to_numeric(
        mast[col],
        errors="coerce"
    )

mast = mast.rename(
    columns={
        "mm_wind_direction_3": "mast_wind_from_deg",
        "mm_wind_speed_3": "mast_wind_speed_ms"
    }
)

mast = mast[
    [
        "timestamp",
        "mast_wind_from_deg",
        "mast_wind_speed_ms"
    ]
].copy()

if mast.timestamp.duplicated().any():
    raise ValueError(
        "Duplicate met-mast timestamps."
    )

# Round measured incoming wind direction to the
# nearest 5 degrees, matching Dataset 1.
#
# Examples:
# 298° -> 300°
# 273° -> 275°
# 358° -> 000°

valid_direction = mast.mast_wind_from_deg.between(
    0, 360
)

mast["wind_from_5deg"] = (
    (
        np.floor(
            (
                mast.mast_wind_from_deg.mod(360)
                + 2.5
            ) / 5
        ) * 5
    ) % 360
).where(valid_direction).astype("Int64")

print("Met-mast records:", f"{len(mast):,}")

# -------------------------------------------------------------
# 5. Merge met-mast conditions with SCADA
# -------------------------------------------------------------

merged = scada.merge(
    mast,
    on="timestamp",
    how="left",
    validate="many_to_one"
)

if len(merged) != original_scada_rows:
    raise ValueError(
        "Row count changed during mast merge."
    )

print(
    "SCADA rows with mast direction:",
    f"{merged.wind_from_5deg.notna().sum():,}"
)

# -------------------------------------------------------------
# 6. Load Dataset 1: Wake-direction lookup
# -------------------------------------------------------------

lookup = pd.read_csv(LOOKUP_PATH)

required_lookup = {
    "model",
    "turbine",
    "wind_from_deg",
    "nominal_deficit_pct",
    "max_offset_deficit_pct",
    "wake_category"
}

if not required_lookup.issubset(lookup.columns):
    raise KeyError(
        "Wake lookup missing columns: "
        f"{required_lookup - set(lookup.columns)}"
    )

lookup["turbine"] = lookup.turbine.map(
    normalize_turbine
)

lookup["wind_from_deg"] = pd.to_numeric(
    lookup.wind_from_deg,
    errors="raise"
).astype("Int64")

if lookup.duplicated(
    ["model", "turbine", "wind_from_deg"]
).any():
    raise ValueError(
        "Duplicate turbine/direction/model lookup records."
    )

# Confirm the classification uses our latest thresholds.
def expected_category(deficit):

    return np.where(
        deficit < 2,
        "LOW",
        np.where(
            deficit > 15,
            "HIGH",
            "MODERATE"
        )
    )


lookup["max_offset_deficit_pct"] = pd.to_numeric(
    lookup["max_offset_deficit_pct"],
    errors="raise"
)

if not (
    lookup.wake_category.astype(str).str.upper()
    == expected_category(
        lookup.max_offset_deficit_pct
    )
).all():
    raise ValueError(
        "Lookup classification does not match "
        "Low <2%, Moderate 2–15%, High >15%."
    )

# -------------------------------------------------------------
# 7. Merge Jensen and Gaussian-style wake predictions
# -------------------------------------------------------------

for model in ["jensen", "gaussian_style"]:

    model_lookup = lookup.loc[
        lookup.model.eq(model),
        [
            "turbine",
            "wind_from_deg",
            "nominal_deficit_pct",
            "max_offset_deficit_pct",
            "wake_category"
        ]
    ].copy()

    if model_lookup.empty:
        raise ValueError(
            f"No wake lookup rows for {model}."
        )

    model_lookup = model_lookup.rename(
        columns={
            "wind_from_deg": "wind_from_5deg",

            "nominal_deficit_pct":
                f"{model}_nominal_deficit_pct",

            "max_offset_deficit_pct":
                f"{model}_max_deficit_pct",

            "wake_category":
                f"{model}_wake_category"
        }
    )

    merged = merged.merge(
        model_lookup,
        on=["turbine", "wind_from_5deg"],
        how="left",
        validate="many_to_one"
    )

if len(merged) != original_scada_rows:
    raise ValueError(
        "Wake lookup merge unexpectedly increased rows."
    )

# -------------------------------------------------------------
# 8. Add comparison variables
# -------------------------------------------------------------

# This is a reference-relative difference, NOT confirmed
# anemometer bias. Mast and turbine are at different sites.

merged["scada_minus_mast_ms"] = (
    merged.wind_speed
    - merged.mast_wind_speed_ms
)

merged["valid_comparison"] = (
    merged.wind_speed.between(0, 50)
    & merged.mast_wind_speed_ms.between(3, 25)
    & merged.jensen_wake_category.notna()
)

merged["date"] = merged.timestamp.dt.date

print("\nMERGE COMPLETE")
print("Total SCADA rows:", f"{len(merged):,}")

print(
    "Rows with Jensen wake classification:",
    f"{merged.jensen_wake_category.notna().sum():,}"
)

print(
    "Rows with valid SCADA/mast comparison:",
    f"{merged.valid_comparison.sum():,}"
)

unmatched = merged.loc[
    merged.wind_from_5deg.notna()
    & merged.jensen_wake_category.isna(),
    "turbine"
].drop_duplicates()

if len(unmatched):
    print(
        "Turbines with no corresponding lookup record:",
        unmatched.tolist()
    )

# -------------------------------------------------------------
# 9. Save complete merged dataset
# -------------------------------------------------------------

merged_path = (
    SAVE_DIR / "ALTA10_SCADA_with_wake_categories.parquet"
)

merged.to_parquet(
    merged_path,
    index=False
)

print("\nMerged dataset saved:")
print(merged_path)

# -------------------------------------------------------------
# 10. Compare SCADA behavior by wake category
# -------------------------------------------------------------

comparison = merged.loc[
    merged.valid_comparison
].copy()

comparison["day"] = (
    comparison.timestamp.dt.floor("D")
)

def build_summary(frame):

    result = (
        frame.groupby(
            ["turbine", "jensen_wake_category"],
            observed=True
        )
        .agg(
            observations=("wind_speed", "size"),

            days=("day", "nunique"),

            mean_scada_ms=("wind_speed", "mean"),

            mean_mast_ms=(
                "mast_wind_speed_ms", "mean"
            ),

            mean_scada_minus_mast_ms=(
                "scada_minus_mast_ms", "mean"
            ),

            median_scada_minus_mast_ms=(
                "scada_minus_mast_ms", "median"
            ),

            original_power_available_n=(
                "real_power_ac", "count"
            ),

            mean_original_power=(
                "real_power_ac", "mean"
            )
        )
        .reset_index()
    )

    return result.sort_values(
        ["turbine", "jensen_wake_category"]
    )


summary_all = build_summary(comparison)

summary_all.to_csv(
    SAVE_DIR / "SCADA_comparison_by_wake_category.csv",
    index=False
)

# -------------------------------------------------------------
# 11. More comparable wind-speed range: mast 7–10 m/s
# -------------------------------------------------------------

comparison_7to10 = comparison.loc[
    comparison.mast_wind_speed_ms.between(7, 10)
]

summary_7to10 = build_summary(
    comparison_7to10
)

summary_7to10.to_csv(
    SAVE_DIR / "SCADA_comparison_mast_7to10_ms.csv",
    index=False
)

# -------------------------------------------------------------
# 12. Display WTG159 and WTG172
# -------------------------------------------------------------

focus_turbines = [
    "WTG159",
    "WTG172"
]

print("\n" + "=" * 65)
print("WTG159 AND WTG172 — ALL VALID MAST CONDITIONS")
print("=" * 65)

display(
    summary_all.loc[
        summary_all.turbine.isin(focus_turbines)
    ].round(3)
)

print("\n" + "=" * 65)
print("WTG159 AND WTG172 — MAST 7–10 m/s")
print("=" * 65)

display(
    summary_7to10.loc[
        summary_7to10.turbine.isin(focus_turbines)
    ].round(3)
)

print("\nMERGED DATASET PREVIEW")

display(
    merged[
        [
            "timestamp",
            "turbine",
            "wind_speed",
            "mast_wind_speed_ms",
            "mast_wind_from_deg",
            "jensen_max_deficit_pct",
            "jensen_wake_category",
            "gaussian_style_wake_category",
            "scada_minus_mast_ms"
        ]
    ].dropna(
        subset=["jensen_wake_category"]
    ).head(15).round(3)
)

print("\n" + "=" * 65)
print("SCADA + WAKE ANALYSIS COMPLETE")
print("=" * 65)

print("\nFiles saved to:")
print(SAVE_DIR)


SCADA records: 1,833,846
Turbines: 47
Met-mast records: 39,306
SCADA rows with mast direction: 1,789,760

MERGE COMPLETE
Total SCADA rows: 1,833,846
Rows with Jensen wake classification: 1,789,760
Rows with valid SCADA/mast comparison: 1,617,003

Merged dataset saved:
C:\Users\alex-\OneDrive\Documents\GitHub\CU-Capstone-Project\data\processed\12_farmwide_wake_exposure\scada_wake_comparison\ALTA10_SCADA_with_wake_categories.parquet

WTG159 AND WTG172 — ALL VALID MAST CONDITIONS


,turbine,jensen_wake_category,observations,days,mean_scada_ms,mean_mast_ms,mean_scada_minus_mast_ms,median_scada_minus_mast_ms,original_power_available_n,mean_original_power
19,WTG159,HIGH,9751,189,4.028,5.429,-1.401,-1.329,6154,265.167
20,WTG159,LOW,25015,244,12.386,11.630,0.755,0.310,17618,1784.546
21,WTG159,MODERATE,906,115,4.490,5.551,-1.062,-1.236,566,522.605
55,WTG172,HIGH,8373,198,4.082,5.773,-1.691,-1.656,5312,312.731
56,WTG172,LOW,25994,256,10.133,11.343,-1.210,-1.412,21574,1537.812
57,WTG172,MODERATE,1461,103,3.970,4.771,-0.801,-0.906,981,239.497



WTG159 AND WTG172 — MAST 7–10 m/s


,turbine,jensen_wake_category,observations,days,mean_scada_ms,mean_mast_ms,mean_scada_minus_mast_ms,median_scada_minus_mast_ms,original_power_available_n,mean_original_power
19,WTG159,HIGH,1606,92,5.896,8.199,-2.303,-2.527,1431,502.848
20,WTG159,LOW,6435,219,8.599,8.608,-0.010,-0.190,5297,1152.449
21,WTG159,MODERATE,94,29,7.352,8.046,-0.694,-0.755,93,931.368
54,WTG172,HIGH,1771,103,5.701,8.207,-2.507,-2.882,1613,463.671
55,WTG172,LOW,6381,215,7.661,8.615,-0.954,-1.068,5875,957.462
56,WTG172,MODERATE,33,6,7.922,7.642,0.280,0.158,33,1081.583



MERGED DATASET PREVIEW


C:\Users\alex-\AppData\Local\Temp\ipykernel_240940\2091846336.py:600: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).head(15).round(3)


,timestamp,turbine,wind_speed,mast_wind_speed_ms,mast_wind_from_deg,jensen_max_deficit_pct,jensen_wake_category,gaussian_style_wake_category,scada_minus_mast_ms
0,2026-01-01 08:00:00+00:00,WTG150,6.908,5.399,67.72,21.435,HIGH,HIGH,1.508
1,2026-01-01 08:00:00+00:00,WTG151,5.718,5.399,67.72,21.944,HIGH,HIGH,0.318
2,2026-01-01 08:00:00+00:00,WTG152,5.837,5.399,67.72,22.731,HIGH,HIGH,0.437
3,2026-01-01 08:00:00+00:00,WTG153,5.952,5.399,67.72,22.593,HIGH,HIGH,0.552
4,2026-01-01 08:00:00+00:00,WTG154,4.758,5.399,67.72,23.045,HIGH,HIGH,-0.642
5,2026-01-01 08:00:00+00:00,WTG155,5.572,5.399,67.72,23.452,HIGH,HIGH,0.173
6,2026-01-01 08:00:00+00:00,WTG156,5.210,5.399,67.72,22.944,HIGH,HIGH,-0.189
7,2026-01-01 08:00:00+00:00,WTG157,5.527,5.399,67.72,24.997,HIGH,HIGH,0.127
8,2026-01-01 08:00:00+00:00,WTG158,5.179,5.399,67.72,25.632,HIGH,HIGH,-0.220
9,2026-01-01 08:00:00+00:00,WTG159,4.060,5.399,67.72,27.563,HIGH,HIGH,-1.339



SCADA + WAKE ANALYSIS COMPLETE

Files saved to:
C:\Users\alex-\OneDrive\Documents\GitHub\CU-Capstone-Project\data\processed\12_farmwide_wake_exposure\scada_wake_comparison


In [4]:

# ============================================================
# ALTA10 — READ ALL SCADA + WAKE COMPARISON DATASETS
# ============================================================

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# ------------------------------------------------------------
# 1. Set folder location
# ------------------------------------------------------------

FOLDER = Path(
    r"C:\Users\alex-\OneDrive\Documents\GitHub"
    r"\CU-Capstone-Project\data\processed"
    r"\12_farmwide_wake_exposure\scada_wake_comparison"
)

# ------------------------------------------------------------
# 2. Dataset 1 — Complete SCADA + wake observations
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DATASET 1 — SCADA + WAKE CATEGORIES")
print("=" * 75)

path1 = FOLDER / "ALTA10_SCADA_with_wake_categories.parquet"

if not path1.exists():
    raise FileNotFoundError(path1)

# Read a preview rather than loading potentially millions
# of SCADA observations into memory.

parquet_file = pq.ParquetFile(path1)

print(f"Total observations: {parquet_file.metadata.num_rows:,}")
print(f"Total columns: {len(parquet_file.schema.names)}")

print("\nColumn names:")
print(parquet_file.schema.names)

scada_wake_preview = next(
    parquet_file.iter_batches(batch_size=20)
).to_pandas()

print("\nFirst 20 observations:")
display(scada_wake_preview)

# ------------------------------------------------------------
# 3. Dataset 2 — All wind-speed conditions
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DATASET 2 — SCADA COMPARISON BY WAKE CATEGORY")
print("=" * 75)

path2 = FOLDER / "SCADA_comparison_by_wake_category.csv"

if not path2.exists():
    raise FileNotFoundError(path2)

wake_summary_all = pd.read_csv(path2)

print(f"Rows: {len(wake_summary_all):,}")
print(f"Columns: {len(wake_summary_all.columns)}")

print("\nColumn names:")
print(wake_summary_all.columns.tolist())

display(wake_summary_all.head(20).round(3))

# ------------------------------------------------------------
# 4. Dataset 3 — Mast wind speed 7–10 m/s
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DATASET 3 — MAST WIND SPEED 7–10 M/S")
print("=" * 75)

path3 = FOLDER / "SCADA_comparison_mast_7to10_ms.csv"

if not path3.exists():
    raise FileNotFoundError(path3)

wake_summary_7to10 = pd.read_csv(path3)

print(f"Rows: {len(wake_summary_7to10):,}")
print(f"Columns: {len(wake_summary_7to10.columns)}")

print("\nColumn names:")
print(wake_summary_7to10.columns.tolist())

display(wake_summary_7to10.head(20).round(3))

# ------------------------------------------------------------
# 5. Focus on WTG159 and WTG172
# ------------------------------------------------------------

focus = ["WTG159", "WTG172"]

columns_to_show = [
    "turbine",
    "jensen_wake_category",
    "observations",
    "days",
    "mean_scada_ms",
    "mean_mast_ms",
    "mean_scada_minus_mast_ms",
    "original_power_available_n",
    "mean_original_power"
]

print("\n" + "=" * 75)
print("WTG159 AND WTG172 — ALL WIND SPEEDS")
print("=" * 75)

display(
    wake_summary_all.loc[
        wake_summary_all.turbine.isin(focus),
        columns_to_show
    ].round(3)
)

print("\n" + "=" * 75)
print("WTG159 AND WTG172 — MAST 7–10 M/S")
print("=" * 75)

display(
    wake_summary_7to10.loc[
        wake_summary_7to10.turbine.isin(focus),
        columns_to_show
    ].round(3)
)

# ------------------------------------------------------------
# 6. Optional: load the full Parquet file into a dataframe
# ------------------------------------------------------------

# Uncomment this line if you want every observation available
# for additional analysis. It may use substantial memory.
#
# scada_wake_full = pd.read_parquet(path1)

print("\n" + "=" * 75)
print("ALL THREE FILES REVIEWED")
print("=" * 75)

print("\nDataFrames available:")
print("scada_wake_preview")
print("wake_summary_all")
print("wake_summary_7to10")

print("\nSource folder:", FOLDER)



DATASET 1 — SCADA + WAKE CATEGORIES
Total observations: 1,833,846
Total columns: 19

Column names:
['timestamp', 'turbine', 'wind_speed', 'KPI_wind_speed', 'real_power_ac', 'KPI_real_power_ac', 'air_density', 'mast_wind_from_deg', 'mast_wind_speed_ms', 'wind_from_5deg', 'jensen_nominal_deficit_pct', 'jensen_max_deficit_pct', 'jensen_wake_category', 'gaussian_style_nominal_deficit_pct', 'gaussian_style_max_deficit_pct', 'gaussian_style_wake_category', 'scada_minus_mast_ms', 'valid_comparison', 'date']

First 20 observations:


,timestamp,turbine,wind_speed,KPI_wind_speed,real_power_ac,KPI_real_power_ac,air_density,mast_wind_from_deg,mast_wind_speed_ms,wind_from_5deg,jensen_nominal_deficit_pct,jensen_max_deficit_pct,jensen_wake_category,gaussian_style_nominal_deficit_pct,gaussian_style_max_deficit_pct,gaussian_style_wake_category,scada_minus_mast_ms,valid_comparison,date
0,2026-01-01 08:00:00+00:00,WTG150,6.907681,6.906036,670.286714,672.60510,1.092081,67.719853,5.399412,70,19.654797,21.434975,HIGH,17.237529,20.540891,HIGH,1.508270,True,2026-01-01
1,2026-01-01 08:00:00+00:00,WTG151,5.717620,5.730449,485.492347,485.57794,1.087094,67.719853,5.399412,70,20.395801,21.943931,HIGH,19.738447,24.797429,HIGH,0.318208,True,2026-01-01
2,2026-01-01 08:00:00+00:00,WTG152,5.836812,5.862610,496.384967,497.79605,1.090355,67.719853,5.399412,70,16.048759,22.730986,HIGH,17.305339,18.330711,HIGH,0.437400,True,2026-01-01
3,2026-01-01 08:00:00+00:00,WTG153,5.951737,5.969328,422.188638,423.57132,1.044950,67.719853,5.399412,70,16.165140,22.593131,HIGH,17.803563,20.431986,HIGH,0.552325,True,2026-01-01
4,2026-01-01 08:00:00+00:00,WTG154,4.757731,4.760764,274.276035,274.52768,1.088453,67.719853,5.399412,70,21.310550,23.045045,HIGH,19.306890,20.464390,HIGH,-0.641681,True,2026-01-01
5,2026-01-01 08:00:00+00:00,WTG155,5.572249,5.551334,231.735037,230.26797,1.089179,67.719853,5.399412,70,21.137477,23.452099,HIGH,18.831696,19.121888,HIGH,0.172837,True,2026-01-01
6,2026-01-01 08:00:00+00:00,WTG156,5.210449,5.196425,376.544799,375.65683,1.084734,67.719853,5.399412,70,15.885147,22.943993,HIGH,18.948327,27.544503,HIGH,-0.188963,True,2026-01-01
7,2026-01-01 08:00:00+00:00,WTG157,5.526678,5.521146,302.057403,300.78232,1.100218,67.719853,5.399412,70,16.598143,24.996507,HIGH,17.866982,25.159019,HIGH,0.127266,True,2026-01-01
8,2026-01-01 08:00:00+00:00,WTG158,5.179184,5.178299,303.971661,302.60312,1.085794,67.719853,5.399412,70,23.522877,25.631731,HIGH,24.112928,30.926952,HIGH,-0.220228,True,2026-01-01
9,2026-01-01 08:00:00+00:00,WTG159,4.060334,4.061986,154.063950,153.17737,1.089676,67.719853,5.399412,70,17.551293,27.562532,HIGH,20.811761,22.707845,HIGH,-1.339078,True,2026-01-01



DATASET 2 — SCADA COMPARISON BY WAKE CATEGORY
Rows: 130
Columns: 10

Column names:
['turbine', 'jensen_wake_category', 'observations', 'days', 'mean_scada_ms', 'mean_mast_ms', 'mean_scada_minus_mast_ms', 'median_scada_minus_mast_ms', 'original_power_available_n', 'mean_original_power']


,turbine,jensen_wake_category,observations,days,mean_scada_ms,mean_mast_ms,mean_scada_minus_mast_ms,median_scada_minus_mast_ms,original_power_available_n,mean_original_power
0,WTG150,HIGH,5794,107,4.206,4.899,-0.693,-0.696,2486,308.037
1,WTG150,LOW,30169,259,12.001,10.721,1.280,1.297,24230,1961.861
2,WTG151,HIGH,9523,182,3.804,5.538,-1.734,-1.519,4954,340.585
3,WTG151,LOW,26277,255,12.409,11.336,1.072,1.303,21492,2075.523
4,WTG152,HIGH,9715,183,3.946,5.497,-1.551,-1.418,5384,346.461
5,WTG152,LOW,24736,241,12.552,11.385,1.166,1.367,20121,2054.438
6,WTG153,HIGH,9808,179,4.413,5.430,-1.017,-0.913,5812,345.772
7,WTG153,LOW,26160,248,11.817,11.414,0.403,0.570,23810,1964.999
8,WTG154,HIGH,10337,183,3.303,5.440,-2.137,-1.883,4334,337.281
9,WTG154,LOW,25516,241,10.880,11.546,-0.665,-0.565,21807,1881.313



DATASET 3 — MAST WIND SPEED 7–10 M/S
Rows: 129
Columns: 10

Column names:
['turbine', 'jensen_wake_category', 'observations', 'days', 'mean_scada_ms', 'mean_mast_ms', 'mean_scada_minus_mast_ms', 'median_scada_minus_mast_ms', 'original_power_available_n', 'mean_original_power']


,turbine,jensen_wake_category,observations,days,mean_scada_ms,mean_mast_ms,mean_scada_minus_mast_ms,median_scada_minus_mast_ms,original_power_available_n,mean_original_power
0,WTG150,HIGH,304,22,7.475,7.992,-0.517,-0.265,221,969.731
1,WTG150,LOW,7903,222,8.763,8.544,0.219,0.027,7247,1309.089
2,WTG151,HIGH,1642,91,5.392,8.193,-2.801,-3.147,1505,467.563
3,WTG151,LOW,6463,220,8.793,8.609,0.183,0.236,5744,1371.816
4,WTG152,HIGH,1629,91,5.718,8.196,-2.478,-2.856,1498,510.006
5,WTG152,LOW,6158,208,8.848,8.597,0.251,0.352,5494,1355.830
6,WTG153,HIGH,1542,88,6.268,8.205,-1.937,-2.139,1515,569.489
7,WTG153,LOW,6666,221,8.269,8.597,-0.328,-0.151,6482,1150.517
8,WTG154,HIGH,1648,91,4.750,8.197,-3.446,-3.764,1497,404.619
9,WTG154,LOW,6486,220,7.276,8.605,-1.329,-1.156,5998,957.279



WTG159 AND WTG172 — ALL WIND SPEEDS


,turbine,jensen_wake_category,observations,days,mean_scada_ms,mean_mast_ms,mean_scada_minus_mast_ms,original_power_available_n,mean_original_power
19,WTG159,HIGH,9751,189,4.028,5.429,-1.401,6154,265.167
20,WTG159,LOW,25015,244,12.386,11.630,0.755,17618,1784.546
21,WTG159,MODERATE,906,115,4.490,5.551,-1.062,566,522.605
55,WTG172,HIGH,8373,198,4.082,5.773,-1.691,5312,312.731
56,WTG172,LOW,25994,256,10.133,11.343,-1.210,21574,1537.812
57,WTG172,MODERATE,1461,103,3.970,4.771,-0.801,981,239.497



WTG159 AND WTG172 — MAST 7–10 M/S


,turbine,jensen_wake_category,observations,days,mean_scada_ms,mean_mast_ms,mean_scada_minus_mast_ms,original_power_available_n,mean_original_power
19,WTG159,HIGH,1606,92,5.896,8.199,-2.303,1431,502.848
20,WTG159,LOW,6435,219,8.599,8.608,-0.010,5297,1152.449
21,WTG159,MODERATE,94,29,7.352,8.046,-0.694,93,931.368
54,WTG172,HIGH,1771,103,5.701,8.207,-2.507,1613,463.671
55,WTG172,LOW,6381,215,7.661,8.615,-0.954,5875,957.462
56,WTG172,MODERATE,33,6,7.922,7.642,0.280,33,1081.583



ALL THREE FILES REVIEWED

DataFrames available:
scada_wake_preview
wake_summary_all
wake_summary_7to10

Source folder: C:\Users\alex-\OneDrive\Documents\GitHub\CU-Capstone-Project\data\processed\12_farmwide_wake_exposure\scada_wake_comparison
